# QAMP AI runs: H10 and H13 (after the freeze)

Runs the two pre-registered AI experiments with the **frozen code** (tag `quantumclock-v1.0-frozen`):

* **H10** (RQ1): the knowledge-grounded AI classifier against the rules. Confirmed if AI F1 > rules F1 and at most 5% of
  AI entries are dropped for invalid citations.
* **H13** (RQ4): AI rewording of the system's explanations. Confirmed if at least 95% of reworded sentences keep valid
  citations and pass the faithfulness check.

**Model (decision D13):** chosen on the run day. The notebook asks the API which models the key can use and takes the
newest balanced ("sonnet") model. Its name is **not printed or saved**; every output records only its SHA-256
fingerprint. The provider's usage page shows the model used.

**Network wrapper:** calls that fail because the service is busy (HTTP 429, 500, 502, 503, 529 or a timeout) are retried
with waits. The frozen code is not changed; the wrapper sits around Python's network call.

**Steps**
1. *Runtime → Run all*.
2. Paste your Anthropic API key (hidden, starts with `sk-ant-api`). It is not saved.
3. About 175 calls, 20-40 minutes, a few US dollars.
4. `qamp_ai_runs.zip` downloads at the end. Delete the API key in the console afterwards.


In [ ]:
# Cell 1: get the frozen code (nothing to edit)
REPO_URL = "https://github.com/FahdSaif/RWA-Asset-tokenization.git"
REPO_REF = "quantumclock-v1.0-frozen"

import subprocess, os
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
commit = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("frozen tag", REPO_REF, "commit", commit)
assert commit.startswith("cb1c560"), "not the frozen commit"


In [ ]:
# Cell 2: key, model and network wrapper (nothing to edit)
import getpass, hashlib, json, os, urllib.request, urllib.error
from pathlib import Path

key = getpass.getpass("Anthropic API key (hidden): ").strip()
if not key.startswith("sk-ant-"):
    raise SystemExit("This is not an Anthropic API key (it should start with sk-ant-). Make one at console.anthropic.com.")

# ask which models this key can use; take the newest 'sonnet' model
req = urllib.request.Request("https://api.anthropic.com/v1/models?limit=100",
                             headers={"x-api-key": key, "anthropic-version": "2023-06-01"})
try:
    models = json.loads(urllib.request.urlopen(req, timeout=30).read())["data"]
except urllib.error.HTTPError as e:
    raise SystemExit(f"The API refused the key ({e.code}). Check the key in the console.") from None
pick = [m for m in models if "sonnet" in m["id"]] or models
pick.sort(key=lambda m: m.get("created_at", ""), reverse=True)
model = pick[0]["id"]
tag = "model-sha256:" + hashlib.sha256(model.encode()).hexdigest()
print(len(models), "models available; using the newest balanced one:", tag)

# network wrapper: retry when the service is busy (loaded into the qamp process; frozen code unchanged)
Path("wrap").mkdir(exist_ok=True)
Path("wrap/sitecustomize.py").write_text('''
import time, urllib.error, urllib.request
_open = urllib.request.urlopen
def _retry(req, *a, **k):
    for i in range(8):
        try:
            return _open(req, *a, **k)
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 529) or i == 7:
                raise
        except (TimeoutError, urllib.error.URLError):
            if i == 7:
                raise
        time.sleep(min(60, 5 * 2 ** i))
urllib.request.urlopen = _retry
''')
print("ready")


In [ ]:
# Cell 3: run H10 and H13 (nothing to edit; run once)
import datetime as dt, subprocess, sys, zipfile

root = Path("repo/rwa-quantumclock")
env = dict(os.environ, PYTHONPATH=str((root / "src").resolve()) + ":" + str(Path("wrap").resolve()),
           ANTHROPIC_API_KEY=key, QAMP_LLM_MODEL=model)
started = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def run(*args):
    r = subprocess.run([sys.executable, "-m", "qamp", *args], cwd=root, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:].replace(model, tag)); raise SystemExit("run failed")

print("H10: AI classifier on 100 tokens (about 20-30 minutes) ...")
run("discover", "--source-dir", "data/sources", "--labels", "data/labels.csv", "--llm")
print("H13: AI rewording of explanations (about 5-10 minutes) ...")
run("explain-check", "--llm")

# replace the model name with its fingerprint in every output
outs = [root / "results/discovery" / f for f in ("discovery_llm.json", "inventory_llm.json", "cbom_llm.json", "llm_log.jsonl")]
outs += [root / "results/explain" / f for f in ("checks.json", "llm_log.jsonl")]
for p in outs:
    if p.exists():
        p.write_text(p.read_text().replace(model, tag))

# decide H10 and H13 with the pre-registered rules (docs/PREREGISTRATION.md section 2)
rules = json.loads((root / "results/discovery/discovery_rules.json").read_text())
ai = json.loads((root / "results/discovery/discovery_llm.json").read_text())
f1_rules, f1_ai = rules["score"]["overall"]["f1"], ai["score"]["overall"]["f1"]
dropped, kept = ai.get("dropped_entries", 0), ai["entries"]
drop_share = dropped / (kept + dropped) if kept + dropped else 1.0
h10 = "CONFIRMED" if (f1_ai > f1_rules and drop_share <= 0.05) else "REFUTED"
llm = json.loads((root / "results/explain/checks.json").read_text())["llm"]
h13 = "CONFIRMED" if (llm.get("retained") or 0) >= 0.95 else "REFUTED"
decisions = dict(
    H10=dict(outcome=h10, ai_f1=f1_ai, rules_f1=f1_rules, ai_precision=ai["score"]["overall"]["precision"],
             ai_recall=ai["score"]["overall"]["recall"], dropped_entries=dropped, kept_entries=kept,
             dropped_share=drop_share, hallucinated_citations=ai.get("hallucinated_citations"),
             per_distinct_code=ai.get("score_per_distinct_code", {}).get("overall")),
    H13=dict(outcome=h13, retained=llm.get("retained"), sentences=llm.get("sentences")),
    model=tag, frozen_commit=commit, started_utc=started,
    finished_utc=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))
out = root / "results/ai_runs"
out.mkdir(parents=True, exist_ok=True)
(out / "decisions.json").write_text(json.dumps(decisions, indent=1))
del key, model, env
print(json.dumps(decisions, indent=1))

with zipfile.ZipFile("qamp_ai_runs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in outs + [out / "decisions.json"]:
        if p.exists():
            z.write(p, p.relative_to(root))
try:
    from google.colab import files; files.download("qamp_ai_runs.zip")
except ImportError:
    print("wrote qamp_ai_runs.zip")
